# Upload Model I/O As A Dataset

This Colab-style notebook saves model inputs and outputs as JSONL, then uploads the file to Burstchester as a dataset.

In [ ]:
#@title 1. Dataset upload settings
REPO_URL = "https://github.com/tomongoose/burstchester.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
REPO_DIR = "/content/burstchester" #@param {type:"string"}
QUESTIONS_PATH = "/content/model-io-questions.jsonl" #@param {type:"string"}
DATASET_PATH = "/content/model-io-dataset.jsonl" #@param {type:"string"}
DATASET_TITLE = "Gemma model IO sample" #@param {type:"string"}
DATASET_DESCRIPTION = "Model input/output pairs collected for supervised fine-tuning." #@param {type:"string"}
DATASET_TAGS = "model-io,sft,gemma" #@param {type:"string"}
SOURCE_MODEL = "google/gemma-4-E2B" #@param {type:"string"}
BASE_MODEL_HINT = "google/gemma-4-E2B" #@param {type:"string"}
TASK_TYPE = "chat" #@param {type:"string"}
LANGUAGE = "en" #@param {type:"string"}
LICENSE = "cc-by-4.0" #@param {type:"string"}
POINT_COST = "10" #@param {type:"string"}
MAX_NEW_TOKENS = 96 #@param {type:"integer"}

print("Settings loaded. Secrets are requested in the next cell.")


In [ ]:
# 2. Load secrets without saving them in notebook output.
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def load_secret(name, prompt, required=True):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(prompt)
    if required and not str(value or "").strip():
        raise ValueError(f"{name} is required.")
    if value:
        os.environ[name] = str(value).strip()
    return os.environ.get(name, "")

load_secret("BURSTCHESTER_ACCESS_TOKEN", "Burstchester access token: ")
load_secret("HF_TOKEN", "Hugging Face token, required for gated/source models: ")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("Secrets configured in environment.")


In [ ]:
# 3. Clone or update the repository.
from pathlib import Path
import subprocess

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

subprocess.run(["git", "checkout", REPO_BRANCH], cwd=repo_dir, check=True)
%cd {REPO_DIR}

In [ ]:
# 4. Install CLI and model inference dependencies.
import subprocess

subprocess.run([
    "python", "-m", "pip", "install", "-q", "-U",
    "transformers",
    "accelerate",
    "huggingface_hub",
], check=True)
print("Model inference dependencies installed.")


In [ ]:
# 5. Download the source model and run one smoke generation.
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. In Colab, open Runtime > Change runtime type and select a GPU.")

tokenizer = AutoTokenizer.from_pretrained(
    SOURCE_MODEL,
    token=os.environ.get("HF_TOKEN") or None,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    SOURCE_MODEL,
    token=os.environ.get("HF_TOKEN") or None,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

def model_generate(prompt):
    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = prompt
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(MAX_NEW_TOKENS),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("Smoke answer:", model_generate("Write one short sentence about dataset quality."))
print("Source model downloaded and ready:", SOURCE_MODEL)


In [ ]:
# 6. Save questions to a JSONL file.
import json
from pathlib import Path

questions = [
    {"id": "q01", "question": "Write a short product tagline for an AI dataset platform."},
    {"id": "q02", "question": "Explain why dataset quality matters when fine-tuning a Gemma model."},
    {"id": "q03", "question": "List three checks before uploading a dataset for supervised fine-tuning."},
    {"id": "q04", "question": "Turn this feature idea into a concise product requirement: users can sell useful datasets for points."},
    {"id": "q05", "question": "Explain the difference between a dataset card and a model card in a marketplace."},
    {"id": "q06", "question": "Write a polite warning shown before spending points to download a dataset."},
    {"id": "q07", "question": "Summarize why model input/output logs can become a useful training dataset."},
    {"id": "q08", "question": "Create a checklist for reviewing whether an uploaded dataset contains sensitive information."},
    {"id": "q09", "question": "Describe how access tokens help connect CLI workflows to a web account."},
    {"id": "q10", "question": "Write a short README paragraph for a legal briefing dataset."},
]

questions_path = Path(QUESTIONS_PATH)
questions_path.parent.mkdir(parents=True, exist_ok=True)
with questions_path.open("w", encoding="utf-8") as f:
    for question in questions:
        f.write(json.dumps(question, ensure_ascii=False) + "\n")

print(f"Wrote {len(questions)} questions:", questions_path)


In [ ]:
# 7. Read the questions file, preview two examples, and save all model I/O pairs.
import json
from pathlib import Path

questions_path = Path(QUESTIONS_PATH)
loaded_questions = [
    json.loads(line)
    for line in questions_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

def generate_answer(question):
    return model_generate(question["question"])

samples = []
for question in loaded_questions:
    answer = generate_answer(question)
    samples.append({
        "messages": [
            {"role": "user", "content": question["question"]},
            {"role": "assistant", "content": answer},
        ]
    })

print("Previewing 2 examples:")
for sample in samples[:2]:
    print(json.dumps(sample, ensure_ascii=False, indent=2))

dataset_path = Path(DATASET_PATH)
dataset_path.parent.mkdir(parents=True, exist_ok=True)
with dataset_path.open("w", encoding="utf-8") as f:
    for sample in samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Wrote {len(samples)} model I/O samples:", dataset_path)


In [ ]:
# 8. Upload the JSONL file as a Burstchester dataset.
import subprocess

upload_cmd = [
    "node", "cli/src/cli.mjs", "upload-test-dataset",
    "--file", DATASET_PATH,
    "--title", DATASET_TITLE,
    "--description", DATASET_DESCRIPTION,
    "--tags", DATASET_TAGS,
    "--source-model", SOURCE_MODEL,
    "--base-model-hint", BASE_MODEL_HINT,
    "--task-type", TASK_TYPE,
    "--language", LANGUAGE,
    "--license", LICENSE,
    "--point-cost", POINT_COST,
]

print("Running:", " ".join(upload_cmd))
subprocess.run(upload_cmd, check=True)


This notebook is self-contained for Colab: it installs the CLI/runtime dependencies, downloads the configured source model, generates answers for the saved question file, writes the model I/O JSONL dataset, and uploads it. To capture an external model service instead, run `proxy-record` in a separate terminal and upload the captured file with `upload-proxy-log`.